In [ ]:
import pypsa 
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
from scripts._helpers import (
    configure_logging,
    get_snapshots,
    load_cutout,
    set_scenario_config,
)
from clusters.add_pointsource_cluster import *

**Set Up**

In [ ]:
fn = 'resources/Iberic5_test/networks/base_s_5__12h_2050.nc'



In [ ]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.iberic5.yaml").read_text())
industrial_production = pd.read_csv("resources/Iberic5_test/industrial_production_base_s_5_2050.csv", index_col=0)
industry_sector_ratios = pd.read_csv("resources/Iberic5_test/industry_sector_ratios.csv", index_col=0)



In [ ]:
n.links.loc[n.links.index.str.contains('cluster')]

In [ ]:
p = Path(fn)  
try:
   if p.exists():
       p.unlink()
       print(f"Deleted {p}")
   else:
       print(f"File not found: {p}")
except Exception as e:
   print(f"Failed to delete {p}: {e}")

**Options**

In [ ]:
ongrid=False
cluster_seq=False
cluster_cost_reduction=0.25
cluster_size=1000                                        #MW, it's the maximum installable capacity for each renewable in renewables in each country cluster
renewables={"solar-hsat","solar","onwind"}
co2_storage_treshold=500*1e3                            #ktonnes CO2 stored from industry threshold for making the node eligible for carbon clusters

**CO2 Use Availability from Industries and Node Definition**

In [ ]:
industrial_production, industry_sector_ratios = check_industrial_production_columns(industrial_production, industry_sector_ratios)

In [ ]:
nodes_with_carbon_clusters,  carbon_available_by_industry_by_node, process_emissions_by_industry_by_node  =find_nodes_eligible_for_carbon_clusters(n, industrial_production, industry_sector_ratios, co2_storage_treshold)


In [ ]:
n = assign_co2_bus_to_carbon_clusters(n, nodes_with_carbon_clusters)

**Buses and Generators of the Cluster Addition**

In [ ]:
n = assign_cluster_generators_and_electricity_buses_to_carbon_clusters(n, config, cluster_size, cluster_cost_reduction, renewables, nodes_with_carbon_clusters)

**Links of the Cluster Addition**

In [ ]:
n = add_cluster_links(n, nodes_with_carbon_clusters, cluster_cost_reduction, ongrid, cluster_seq)       

**Storages of the Cluster Addition**

In [ ]:
n = add_cluster_storages(n, nodes_with_carbon_clusters, cluster_cost_reduction)


In [ ]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)

**Printing to Check**

**Exporting**

In [ ]:
n.export_to_netcdf(fn)
